# Aggregating results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
import matplotlib
import seaborn as sns
import numpy as np
import pandas as pd
import glob
import os,sys,json
import pickle
import warnings
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')

working_dir = os.path.abspath("")
sys.path.append(working_dir)
# custom utils
from funcs.utils import *
from funcs.feat_funcs import *
from funcs.metric_funcs import *
custom_palette = ["#5F6C7B", "#368899", "#E03F67", "#F47C6D"]
sns.set_palette(custom_palette)
# my_palette = sns.color_palette(custom_palette)
matplotlib.rcParams['font.weight'] = 'light'

metadata_file = 'databricks/patient_meta_info.csv'

metric_labels = {'auroc_sample':'AUROC',
                  'auprc_sample':'AUPRC',
                  'recall_event':'Seizure Detected, %', # 1-miss
                  'recall_sample':'Seizure Sample Detected, %', # total_sz_dura-miss_min
                  'tn':'Specificity, %', 
                   'precision_event':'Precision, %',
                   'precision_sample':'Precision Sample, %',
                   'fp':'False Alarm/h',
                   'f1_event':'F1 Score',
                   'f1_sample':'F1 Sample',
                   'balanced_acc':'Balanced Accuracy',
                   'mcc':'MCC'}

plot_vars = ['auroc_sample','auprc_sample','recall_event','f1_event','fp']
plot_labels = [metric_labels[k] for k in plot_vars]

ieeg_list = ... # ieeg patients
rm_patients = ... # rejected patients
patient_order = ... # included patients in order
out_dir = ... # output directory
patient_rank = {v: i for i, v in enumerate(patient_order)}

In [ ]:
def plot_paired_line(df1, df2, variables=None, figsize=(15, 6), save_path=None, ieeg=True, ieeg_col='patient'):
    if variables is None:
        variables = df1.columns.tolist()
    fig, axes = plt.subplots(1, len(variables), sharey=True, figsize=figsize, sharex=False)
    if len(variables) == 1:
        axes = [axes]  # make iterable
    ax2 = axes[-1].twinx()
    for ax, var in zip(axes, variables):
        if var == 'fp':
            ax = ax2
        # Plot individual subjects
        for i in range(len(df1)):
            if ieeg:
                if df1[ieeg_col].iloc[i] in ieeg_list:
                    ax.plot([0, 1], [df1[var].iloc[i], df2[var].iloc[i]], 
                            color='#C01711', linewidth=1)
                else:
                    ax.plot([0, 1], [df1[var].iloc[i], df2[var].iloc[i]], 
                            color='k', linewidth=1)
            else:
                ax.plot([0, 1], [df1[var].iloc[i], df2[var].iloc[i]], 
                        color='k', linewidth=1)
            ax.plot([0], [df1[var].iloc[i]], 
                    markerfacecolor='white',    # inside color
                    markeredgecolor="#368899",  # border color
                    markeredgewidth=1.5,
                    marker="o")
            ax.plot([1], [df2[var].iloc[i]], 
                    markerfacecolor='white',    # inside color
                    markeredgecolor="#E03F67",  # border color
                    markeredgewidth=1.5,
                    marker="o")
        ax.set_xticks([0,1])
        ax.set_xticklabels(['Baseline','Fine-tuned'], rotation=45)
        ax2.set_xticklabels(['Baseline','Fine-tuned'], rotation=45)
        ax.set_title(metric_labels[var])
        ax.set_xlim((-0.3,1.3))
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path+'.svg', dpi=300)
        plt.close(fig)

def plot_mean_diff_overlay(long_df, sep_axis_fp = False, color = None, variables = None, ieeg=False, ieeg_col='patient', ieeg_color=None):

    if color is None:
        color = '#2A6F97'
    if ieeg and ieeg_color is None:
        ieeg_color = '#C01711'
    if variables is None:
        variables = plot_vars
        labels = plot_labels
    else:
        labels = [metric_labels[v] for v in variables]
    long_df = long_df[long_df['metric'].isin(variables)]
    n_metric = len(variables)
    fig, ax = plt.subplots(1,n_metric, figsize=(10, 5), sharey=True)
    for e,metric in enumerate(variables):
        if metric == 'fp':
            if sep_axis_fp:
                ax2 = ax[e].twinx()
                ax_use = ax2
            else:
                ax_use = ax[e]
        else:
            ax_use = ax[e]
        sns.boxplot(
            y='value',
            data=long_df[long_df['metric']==metric],
            color='black',
            notch=False, showcaps=True,showfliers=False,
            width=0.6,
            boxprops={"facecolor": "white","edgecolor":"black"},
            medianprops={"color": "black", "linewidth": 2},
            ax=ax_use
        )
        if not ieeg:
            sns.stripplot(
                y='value',
                jitter = 0.06,
                data=long_df[long_df['metric']==metric],
                size=7,
                alpha = 1,
                legend=False,zorder=100,color = color,
                ax = ax_use
            )
        else:
            sns.stripplot(
                y='value',
                jitter = 0.06,
                data=long_df[(long_df['metric']==metric)&(~long_df[ieeg_col].isin(ieeg_list))],
                size=7,
                alpha = 1,
                legend=False,zorder=100,color = color,
                ax = ax_use
            )
            sns.stripplot(
                y='value',
                jitter = 0.06,
                data=long_df[(long_df['metric']==metric)&(long_df[ieeg_col].isin(ieeg_list))],
                size=7,
                alpha = 0.6,
                legend=False,zorder=100,color = ieeg_color,
                ax = ax_use
            )
        ax_use.hlines([0],xmin=-1,xmax=1,linestyle='--',color='gray',linewidth=1,zorder=0)
        ax_use.legend().remove()
        ax_use.set_xlabel('')
        ax_use.set_ylabel('')
        ax_use.set_title(labels[e],fontsize=12)
        ax_use.set_xticks(ticks=[0], labels=['Delta'],fontsize=12, rotation=25)
    return fig, ax

def bootstrap_ci(
    values,
    clusters=None,
    n_boot: int = 1000,
    ci: float = 95,
    seed: int = 42,
):
    """Return the sample mean and a percentile bootstrap CI for the mean.

    If clusters is provided, resample whole clusters with replacement.
    """
    frame = pd.DataFrame({"value": np.asarray(values, dtype=float)})
    if clusters is not None:
        frame["cluster"] = np.asarray(clusters)
    frame = frame.dropna()

    if frame.empty:
        return np.nan, np.nan, np.nan

    mean = float(frame["value"].mean())
    if len(frame) == 1:
        return mean, mean, mean

    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot, dtype=float)

    if clusters is None:
        sample_values = frame["value"].to_numpy()
        for index in range(n_boot):
            boot_means[index] = rng.choice(
                sample_values,
                size=len(sample_values),
                replace=True,
            ).mean()
    else:
        cluster_values = {
            cluster: group["value"].to_numpy()
            for cluster, group in frame.groupby("cluster", sort=False)
        }
        cluster_ids = np.fromiter(cluster_values.keys(), dtype=object)
        for index in range(n_boot):
            sampled_clusters = rng.choice(
                cluster_ids,
                size=len(cluster_ids),
                replace=True,
            )
            boot_sample = np.concatenate(
                [cluster_values[cluster] for cluster in sampled_clusters]
            )
            boot_means[index] = boot_sample.mean()

    alpha = (100 - ci) / 2
    ci_lower = float(np.percentile(boot_means, alpha))
    ci_upper = float(np.percentile(boot_means, 100 - alpha))
    return mean, ci_lower, ci_upper


def summarize_series(
    values,
    clusters=None,
    n_boot: int = 1000,
    ci: float = 95,
    seed: int = 42,
):
    """Summary statistics for one numeric sample."""
    series = pd.Series(values, dtype=float).dropna()
    if series.empty:
        return pd.Series(
            {
                "n": 0,
                "mean": np.nan,
                "std": np.nan,
                "median": np.nan,
                "iqr": np.nan,
                "ci_lower": np.nan,
                "ci_upper": np.nan,
            }
        )

    cluster_values = None
    if clusters is not None:
        cluster_values = pd.Series(clusters, index=series.index).loc[series.index]

    mean, ci_lower, ci_upper = bootstrap_ci(
        series.to_numpy(),
        clusters=None if cluster_values is None else cluster_values.to_numpy(),
        n_boot=n_boot,
        ci=ci,
        seed=seed,
    )

    tmp = pd.Series(
        {
            "n": int(series.shape[0]),
            "mean": mean,
            "std": float(series.std(ddof=1)),
            "median": float(series.median()),
            "q1": float(series.quantile(0.25)),
            "q3": float(series.quantile(0.75)),
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        }
    )
    tmp["mean [SD]"] = f"{tmp['mean']:.3f} [{tmp['std']:.3f}]"
    tmp["median [IQR]"] = f"{tmp['median']:.3f} [{tmp['q1']:.3f},{tmp['q3']:.3f}]"
    tmp["mean [95%CI]"] = f"{tmp['mean']:.3f} [{tmp['ci_lower']:.3f},{tmp['ci_upper']:.3f}]"
    return tmp


def summary(
    data: pd.DataFrame,
    value_cols: list[str] | None = None,
    group_cols: list[str] | None = None,
    cluster_col: str | None = None,
    n_boot: int = 1000,
    ci: float = 95,
    seed: int = 42,
) -> pd.DataFrame:
    """Summarize numeric columns with mean, SD, median, IQR, and bootstrap 95% CI.

    Use cluster_col when observations are nested within patients or admissions.
    """
    value_columns = value_cols if value_cols is not None else list(data.columns)
    group_columns = group_cols if group_cols is not None else []

    missing = [column for column in value_columns + group_columns if column not in data.columns]
    if missing:
        raise KeyError(f"Columns not found in data: {missing}")
    if cluster_col is not None and cluster_col not in data.columns:
        raise KeyError(f"Cluster column not found in data: {cluster_col!r}")

    records: list[dict[str, object]] = []
    if group_columns:
        groups = data.groupby(group_columns, sort=False, dropna=False)
    else:
        groups = [((), data)]

    for keys, group in groups:
        if not isinstance(keys, tuple):
            keys = (keys,)
        for value_col in value_columns:
            clusters = group[cluster_col] if cluster_col is not None else None
            stats = summarize_series(
                group[value_col],
                clusters=clusters,
                n_boot=n_boot,
                ci=ci,
                seed=seed,
            )
            row = {column: keys[index] for index, column in enumerate(group_columns)}
            row['value'] = value_col
            row.update(stats.to_dict())
            records.append(row)

    index_names = group_columns + ['value']
    return pd.DataFrame.from_records(records).set_index(
        index_names
    )


def get_moving_max_prob(sz_probs,win_size=10,gap_size=2):
    perc = gap_size/win_size
    probs = [np.min(sz_probs[i-win_size:i]) for i in range(win_size,len(sz_probs))]
    return np.max(probs) 

def flatten_tableone(df):
    new_col_names = {k:f'{k} (n={v})' for k,v in df.loc['n'].to_dict(orient='records')[0].items() if v} 
    df = df.drop(('n',''),axis=0)
    new_rows = []
    for group in df.index.get_level_values(0).unique():
        if group == 'n':
            continue
        block = df.xs(group, level=0)
        # Insert a blank row with group label
        if 'mean' in group or 'median' in group:
            block.index = [group]
            new_rows.append(block)
        else:
            label_row = pd.DataFrame([[""] * (df.shape[1])], columns=df.columns, index=[group])
            block.index = ['    ' + idx for idx in block.index]
            new_block = pd.concat([label_row, block])
            new_rows.append(new_block)
    flat_df = pd.concat(new_rows)
    flat_df.index.name = None
    flat_df = flat_df.rename(new_col_names,axis=1)
    return flat_df      

### Get metadata

In [ ]:
metadata = pd.read_csv(metadata_file, parse_dates=['enroll_time'])
for e, row in metadata.iterrows():
    rid = row['RID']
    metadata.loc[e,'enroll_time'] = row['enroll_time'].tz_localize(None)
    with open(f"INFO/{rid}.pkl", "rb") as f:
        info = pickle.load(f)
    metadata.loc[e,'first_name'] = info['firstName']
    metadata.loc[e,'last_name'] = info['lastName']
    if info['studyType'] == 'eeg':
        metadata.loc[e,'recording_type'] = 'Scalp'
    elif info['studyType'] == 'ieeg':
        metadata.loc[e,'recording_type'] = 'SEEG'
    # from labels
    label_df = pd.read_pickle(f'labels/{rid}_data_labels.pkl')
    label_df = label_df.reset_index().rename(columns={'index':'timestamps'})
    data_timestamps = label_df['timestamps']
    start_time, end_time = data_timestamps.iloc[0], data_timestamps.iloc[-1]
    metadata.loc[e,'recording_start_time'] = start_time
    metadata.loc[e,'recording_stop_time'] = end_time
    metadata.loc[e,'length_of_stay'] = (end_time-start_time).total_seconds()/3600/24
    metadata.loc[e,'length_of_enrollment'] = (end_time-metadata.loc[e,'enroll_time']).total_seconds()/3600/24
    
    # from annots
    sz_events = pd.read_csv(f'annots/{rid}/seizure_events.csv', parse_dates=['Start','Stop'])
    metadata.loc[e,'num_sz'] = sz_events.shape[0]
    if not sz_events.empty:
        sz_events['Dura'] = (sz_events['Stop']-sz_events['Start']).dt.total_seconds()/60
        metadata.loc[e,'avg_sz_dura'] = sz_events['Dura'].mean()
    sz_events_after_enroll = sz_events[sz_events['Start'] >= metadata.loc[e,'enroll_time']]
    metadata.loc[e,'num_sz_after_enroll'] = sz_events_after_enroll.shape[0]
    push_events = pd.read_csv(f'annots/{rid}/patient_events.csv', parse_dates=['Start','Stop'])
    metadata.loc[e,'num_push'] = push_events.shape[0]
    push_events_after_enroll = push_events[push_events['Start'] >= metadata.loc[e,'enroll_time']]
    metadata.loc[e,'num_push_after_enroll'] = push_events_after_enroll.shape[0]
    
    # from databricks
    try:
        databricks_annots = pd.read_csv(f'annots/{rid}/databricks_events.csv')
        databricks_annots = databricks_annots[~databricks_annots['classified_response'].isna()]
        metadata.loc[e,'num_annots'] = databricks_annots.shape[0]
        metadata.loc[e,'num_yes_annots'] = databricks_annots[databricks_annots['classified_response']=='yes'].shape[0]
        metadata.loc[e,'num_no_annots'] = databricks_annots[databricks_annots['classified_response']=='no'].shape[0]
        metadata.loc[e,'num_uncertain_annots'] = databricks_annots[databricks_annots['classified_response']=='uncertain'].shape[0]
    except FileNotFoundError:
        pass
metadata.to_csv(metadata_file,index=False)

### Create Table1 

In [ ]:
metadata = pd.read_csv('databricks/patient_meta_info.csv')
metadata = metadata[~metadata['RID'].isin(rm_patients)]
fields = {'Age':'Age',
          'Sex':'Sex',
          'EEG Type':'EEG Type',
          'Laterality':'Epilepsy Laterality',
          'Localization':'Epilepsy Localization',
          'length_of_stay':'Length of Stay (days)',
          'num_sz':'Number of Seizures',
          'length_of_enrollment':'Length of Enrollment (days)',
          'num_sz_after_enroll':'Number of Seizures after Enrollment'}

In [ ]:
table1 = TableOne(metadata, columns=list(fields.keys()), 
                  nonnormal=['length_of_stay', 'num_sz', 'length_of_enrollment', 'num_sz_after_enroll'],
                  categorical=['Sex','EEG Type','Laterality','Localization'], 
                  missing=False, htest_name=False, pval=False, decimals=2, rename=fields)
flatten_tableone(table1.tableone).to_csv(f'{out_dir}/table1.csv')

### Baseline model performance

In [ ]:
metadata = pd.read_csv(metadata_file)
metadata = metadata[['RID','Age','Sex','EEG Type','length_of_stay','length_of_enrollment','num_sz','num_sz_after_enroll','num_annots','Surveys','Seizures','Sleep','Data Agent','Spikes']]
metadata = metadata[~metadata['RID'].isin(rm_patients)]

In [ ]:
base_results = glob.glob('results/*/*base_model/metrics/*.csv')
base_results += glob.glob('results/*/oncet_base_model/metrics/*.csv')
results = []
for f in base_results:
    try:
        data = pd.read_csv(f,index_col=0)
        tmp = data[data['setting'] == 'Finetune']
    except:
        data = pd.read_csv(f)
        tmp = data[data['setting'] == 'Finetune']
    tmp['patient'] = f.split('/')[1]
    if not tmp['patient'].iloc[0] in rm_patients:
        results.append(tmp)
results = pd.concat(results,axis=0)[['patient','auroc_sample','auprc_sample','recall_event','recall_sample',
                                    'tn','fp','precision_event','precision_sample','f1_event', 'f1_sample', 'balanced_acc', 'mcc']]
metadata = metadata.merge(results, left_on = 'RID',right_on='patient',how='left').drop(columns='patient')

In [ ]:
rename = {'index':'patient'}
rename.update(metric_labels)
(metadata
     .drop(columns=['RID'])
     .reset_index()
     .rename(columns=rename)
     .to_csv(f'{out_dir}/supp_patient_detailed_info_baseline_performance.csv', index=False))

In [ ]:
results_long = pd.melt(
    results,
    id_vars='patient',
    var_name="metric",        # new column with former column names
    value_name="value"           # new column with values
)
fig,ax=plot_mean_diff_overlay(results_long,sep_axis_fp=True, variables = plot_vars, ieeg=True, ieeg_col='patient')
fig.savefig(f'{out_dir}/supp_fig_baseline_performance_revision.svg',dpi=300)

In [ ]:
summary_tbl = summary(metadata[plot_vars])
summary_tbl.to_csv(f'{out_dir}/supp_summary_baseline_results.csv')

### Results

In [ ]:
# updated correct ver
setting = '_0.5_extend_0.5'
adjth = False
calibrate = False
if calibrate:
    ft_setting = 'Finetune+Calibrate'
    pred_col = 'smoothed_pred_temp'
    prob_col = 'sz_prob_temp'
else:
    ft_setting = 'Finetune'
    pred_col = 'smoothed_pred'
    prob_col = 'sz_prob'
label_type = 'databricks'# or 'patient'
model_code = 'update_3_1.2_fix_unfreeze1'
os.makedirs(f'{out_dir}/full_trajectory',exist_ok=True)
metadata = pd.read_csv(metadata_file, parse_dates=['recording_start_time','recording_stop_time','enroll_time'])
patients = 'scalp'
if patients == 'all':
    patient_ids = sorted(os.listdir('../rawdata'))
elif patients == 'scalp':
    patient_ids = sorted(os.listdir('../rawdata'))
    patient_ids = [p for p in patient_ids if p not in ieeg_list]
elif patients == 'ieeg':
    patient_ids = ieeg_list
else:
    patient_ids = sorted(patients.split(','))

patient_ids = [p for p in patient_ids if p not in rm_patients]

agg_baseline = [] # average performance across finetuned models on test set for baseline model
agg_finetune = [] # average performance across finetuned models on test set for finetuned model
agg_prev = []# average performance across finetuned models on test set for prev finetuned model
mean_diff = [] # for each finetuned model, performance difference on test set between finetuned and baseline
mean_perc_diff = [] # percentage performance difference
full_baseline = [] # full stay performance at baseline
full_finetune = [] # full stay performance with active finetuning
for rid in patient_ids:
    print(rid)
    if rid in rm_patients:
        continue
    # load INFO
    with open(f"INFO/{rid}.pkl", "rb") as f:
        info = pickle.load(f)
    if info['studyType'] == 'eeg':
        feat_setting = {'win':10,'stride':2}
        base_model_name = 'base_model'
    elif info['studyType'] == 'ieeg':
        feat_setting = {'win':1,'stride':1}
        base_model_name = 'oncet_base_model'
    meta_info = metadata[metadata['RID']==rid].iloc[0]

    # load events
    sz_events = pd.read_csv(f'annots/{rid}/seizure_events.csv', parse_dates=['Start','Stop'])
    push_events = pd.read_csv(f'annots/{rid}/patient_events.csv', parse_dates=['Start','Stop']).iloc[1:]
    if not sz_events.empty:
        sz_times = (sz_events['Start']-meta_info['recording_start_time']).dt.total_seconds()/3600/24
    else:
        sz_times = np.array([])
    if not push_events.empty:
        push_times = (push_events['Start']-meta_info['recording_start_time']+pd.Timedelta(seconds=60)).dt.total_seconds()/3600/24
    else:
        push_times = np.array([])
        
    # load base_model prediction
    baseline_pred = pd.read_csv(f'results/{rid}/{base_model_name}/pred/{base_model_name}{setting}.csv', parse_dates=['timestamp'])
    pred_sz_events = extract_seiz_ranges(baseline_pred[pred_col].values)
    pred_sz_events = [{'Start':baseline_pred['timestamp'].iloc[sz[0]]-pd.Timedelta(seconds=feat_setting['stride']),
                      'Stop':baseline_pred['timestamp'].iloc[sz[1]],
                      'is_sz':True if baseline_pred['label'].iloc[sz[0]-30:sz[1]+15].any() else False,
                      'avg_prob':baseline_pred[prob_col].iloc[sz[0]:sz[1]].mean()}
                      for sz in pred_sz_events]
    pred_sz_events = pd.DataFrame(pred_sz_events)
    pred_sz_times = (pred_sz_events['Start']-meta_info['recording_start_time']).dt.total_seconds()/3600/24
    sz_ranges = extract_seiz_ranges(baseline_pred['label'].values)
    sz_events_detected = np.array([np.any(baseline_pred[pred_col].iloc[start-15:end+30]==1) for start, end in sz_ranges])
    
    # version 1, plot predictions
    model = f'{rid}_{base_model_name}_lr5e-06_wd1e-03_{label_type}_{model_code}'
    pred_files = glob.glob(f'results/{rid}/{model}/pred/*.csv')
    if not pred_files:
        continue
    pred_files_df = pd.DataFrame({'filename':pred_files})
    pred_files_df['batch_ind'] = pred_files_df['filename'].apply(lambda x:int(x.split('/')[-1].split('_')[1]))
    pred_files_df['adjth'] = pred_files_df['filename'].apply(lambda x: 1 if 'adjth' in x else 0)
    pred_files_df = pred_files_df.sort_values(['batch_ind', 'adjth'])
    if adjth:
        pred_files_df = pred_files_df.drop_duplicates('batch_ind', keep='last')
    else:
        pred_files_df = pred_files_df.drop_duplicates('batch_ind', keep='first')
    pred_files = pred_files_df['filename'].to_list()
    finetune_inds = pred_files_df['batch_ind'].to_list()#[int(f.split('/')[-1].split('_')[1]) for f in metric_files]
    finetune_inds.append(baseline_pred['batch_id'].max())
    
    # build full pred
    full_pred = []
    comp_pred = [] # for pairwise comparison between finetuned and baseline and previous finetuned
    comp_base = []
    comp_prev = []
    full_pred.append(baseline_pred[baseline_pred['batch_id'].isin(list(range(finetune_inds[0]+1)))])
    for i, f in enumerate(pred_files):
        pred = pd.read_csv(f, parse_dates=['timestamp'])
        comp_pred.append(pred[pred['batch_id'] > finetune_inds[i]])
        comp_base.append(baseline_pred[baseline_pred['batch_id'] > finetune_inds[i]])
        if i == 0:
            comp_prev.append(baseline_pred[baseline_pred['batch_id'] > finetune_inds[i]])
        else:
            comp_prev.append(comp_pred[-1][comp_pred[-1]['batch_id'] > finetune_inds[i]])
        full_pred.append(pred[pred['batch_id'].isin(list(range(finetune_inds[i]+1,finetune_inds[i+1]+1)))])
    full_pred = pd.concat(full_pred, axis=0).sort_values('timestamp')

    pred_sz_events_ft = extract_seiz_ranges(full_pred[pred_col].values)
    pred_sz_events_ft = [{'Start':full_pred['timestamp'].iloc[sz[0]]-pd.Timedelta(seconds=feat_setting['stride']),
                      'Stop':full_pred['timestamp'].iloc[sz[1]],
                      'is_sz':True if full_pred['label'].iloc[sz[0]-30:sz[1]+15].any() else False,
                      'avg_prob':full_pred[prob_col].iloc[sz[0]:sz[1]].mean()}
                      for sz in pred_sz_events_ft]
    pred_sz_events_ft = pd.DataFrame(pred_sz_events_ft)
    pred_sz_times_ft = (pred_sz_events_ft['Start']-meta_info['recording_start_time']).dt.total_seconds()/3600/24
    sz_ranges_ft = extract_seiz_ranges(full_pred['label'].values)
    sz_events_detected_ft = np.array([np.any(full_pred[pred_col].iloc[start-15:end+30]==1) for start, end in sz_ranges_ft])

    fig, axs = plt.subplots(6,1,figsize=(15,10),sharex=True)
    axs[0].vlines(sz_times,ymin=0,ymax=1,color='k')
    axs[0].vlines((meta_info['enroll_time']-meta_info['recording_start_time']).total_seconds()/3600/24,ymin=0,ymax=1,color='r')
    axs[0].set_title('Seizure Times and Enroll Time')
    axs[1].vlines(push_times,ymin=0,ymax=1,color='blue')
    axs[1].set_title('Push Times')
    axs[2].vlines((np.array(finetune_inds[:-1])+1)*3/24,ymin=0,ymax=1,color='k',linestyle='--',alpha=0.5)
    axs[2].set_title('Finetune Times')
    
    axs[4].vlines(pred_sz_times[pred_sz_events['is_sz']==False],ymin=0,ymax=1,color='orange')
    axs[4].vlines(pred_sz_times[pred_sz_events['is_sz']],ymin=0,ymax=1,color='green')
    axs[4].set_title('Baseline Alarms')
    axs[5].vlines(pred_sz_times_ft[pred_sz_events_ft['is_sz']==False],ymin=0,ymax=1,color='orange')
    axs[5].vlines(pred_sz_times_ft[pred_sz_events_ft['is_sz']],ymin=0,ymax=1,color='green')
    axs[5].set_title('Finetuned Alarms')
    
    # version 2, plot summary metrics
    # metrics trajectory
    model = f'{rid}_{base_model_name}_lr5e-06_wd1e-03_{label_type}_{model_code}'
    baseline_metrics = pd.read_csv(f'results/{rid}/{base_model_name}/metrics/{base_model_name}{setting}.csv', index_col=0)
    baseline_metrics = baseline_metrics[baseline_metrics['setting'] == ft_setting]
    metric_files = glob.glob(f'results/{rid}/{model}/metrics/*.csv')
    metric_files = [f for f in metric_files if not f.endswith('full.csv') ]
    metric_files_df = pd.DataFrame({'filename':metric_files})
    metric_files_df['batch_ind'] = metric_files_df['filename'].apply(lambda x:int(x.split('/')[-1].split('_')[1]))
    metric_files_df['adjth'] = metric_files_df['filename'].apply(lambda x: 1 if 'adjth' in x else 0)
    metric_files_df = metric_files_df.sort_values(['batch_ind', 'adjth'])
    if adjth:
        metric_files_df = metric_files_df.drop_duplicates('batch_ind', keep='last')
    else:
        metric_files_df = metric_files_df.drop_duplicates('batch_ind', keep='first')
    metric_files = metric_files_df['filename'].to_list()
    finetune_inds = metric_files_df['batch_ind'].to_list()#[int(f.split('/')[-1].split('_')[1]) for f in metric_files]
    finetune_inds.append(len(baseline_metrics)-1)
    
    # build full metric trajectory 
    full_metric = []
    for batch in full_pred['batch_id'].unique():
        batch_pred = full_pred[full_pred['batch_id']==batch]
        batch_label = batch_pred['label'].values
        batch_prob = batch_pred[prob_col].values
        batch_pred_vals = batch_pred[pred_col].values
        full_metric.append(compute_metrics(batch_label, batch_pred_vals, batch_prob))
    full_metric = pd.DataFrame(full_metric)
    baseline_metric_whole = compute_metrics(baseline_pred['label'].values, baseline_pred[pred_col].values, baseline_pred[prob_col].values)
    finetune_metric_whole = compute_metrics(full_pred['label'].values, full_pred[pred_col].values, full_pred[prob_col].values)
    
    comp_base_metrics = []
    comp_prev_metrics = []
    comp_finetune_metrics = []
    for base,prev,curr in zip(comp_base, comp_prev, comp_pred):
        comp_base_metrics.append(compute_metrics(base['label'].values, base[pred_col].values, base[prob_col].values))
        comp_prev_metrics.append(compute_metrics(prev['label'].values, prev[pred_col].values, prev[prob_col].values))
        comp_finetune_metrics.append(compute_metrics(curr['label'].values, curr[pred_col].values, curr[prob_col].values))
    comp_base_metrics = pd.DataFrame(comp_base_metrics)
    comp_prev_metrics = pd.DataFrame(comp_prev_metrics)
    comp_finetune_metrics = pd.DataFrame(comp_finetune_metrics)
    tmp = comp_base_metrics[plot_vars].mean().to_frame().T
    tmp.index = [rid]
    agg_baseline.append(tmp)
    tmp = comp_prev_metrics[plot_vars].mean().to_frame().T
    tmp.index = [rid]
    agg_prev.append(tmp)
    tmp = comp_finetune_metrics[plot_vars].mean().to_frame().T
    tmp.index = [rid]
    agg_finetune.append(tmp)
    tmp = (comp_finetune_metrics[plot_vars]-comp_base_metrics[plot_vars]).median().to_frame().T
    tmp.index = [rid]
    mean_diff.append(tmp)
    tmp = ((comp_finetune_metrics[plot_vars]-comp_base_metrics[plot_vars])/comp_base_metrics[plot_vars]*100).median().to_frame().T
    tmp.index = [rid]
    mean_perc_diff.append(tmp)

    txt = '\n'.join([f'{metric_labels[var]}: {baseline_metric_whole[var]:.3f}' for var in ['auroc_sample','auprc_sample','recall_event','f1_event','fp']])
    axs[4].text(1.0, 0.1, txt, transform=axs[4].transAxes, fontsize=6,color="k")
    txt = '\n'.join([f'{metric_labels[var]}: {finetune_metric_whole[var]:.3f}' for var in ['auroc_sample','auprc_sample','recall_event','f1_event','fp']])
    axs[5].text(1.0, 0.1, txt, transform=axs[5].transAxes, fontsize=6,color="k")
    axs[3].plot(baseline_metrics.index.astype(int)*3/24, baseline_metrics['fp'].values,label='baseline',color=custom_palette[1], marker='o', markerfacecolor='white')
    axs[3].plot(full_metric.index.astype(int)*3/24, full_metric['fp'].values, label='finetuned', color=custom_palette[2],marker='o', markerfacecolor='white')
    axs[3].set_ylabel(metric_labels['fp'])
    axs[3].legend(loc='upper left')
    axs[5].set_xlabel("Length of Stay (days)")
    fig.savefig(f'{out_dir}/full_trajectory/{rid}_adjth_{adjth}_calibration_{calibrate}_{label_type}_{model_code}.svg',dpi=300)
    plt.close()
    
    baseline_tmp = pd.DataFrame([baseline_metric_whole])[plot_vars]#baseline_metrics[plot_vars].mean().to_frame().T
    baseline_tmp.index = [rid]
    full_baseline.append(baseline_tmp)
    finetune_tmp = pd.DataFrame([finetune_metric_whole])[plot_vars]#full_metric[plot_vars].mean().to_frame().T
    finetune_tmp.index = [rid]
    full_finetune.append(finetune_tmp)
    
    # the agg on test data only
    
full_baseline = pd.concat(full_baseline)
full_finetune = pd.concat(full_finetune)
agg_baseline = pd.concat(agg_baseline)
agg_finetune = pd.concat(agg_finetune)
agg_prev = pd.concat(agg_prev)
mean_diff = pd.concat(mean_diff)
mean_perc_diff = pd.concat(mean_perc_diff)
full_diff = full_finetune-full_baseline
full_perc_diff = ((full_finetune-full_baseline)/full_baseline*100)

In [ ]:
# save scalp results to concatenate with ieeg results
with open(f'{out_dir}/scalp_results_{label_type}.pkl','wb') as f:
    pickle.dump((full_baseline, full_finetune, mean_diff, mean_perc_diff, full_diff, full_perc_diff, agg_baseline, agg_finetune, agg_prev), f)

### Intracranial patients

In [ ]:
setting = '_0.9_0.15'
calibrate = False
adjth = False
ft_setting = 'Finetune'
pred_col = 'smoothed_pred'
prob_col = 'avg_prob'
label_type = 'patient'#'patient'
model_code = 'topk_new_update_rule_unfreeze2'
os.makedirs(f'{out_dir}/full_trajectory',exist_ok=True)
metadata = pd.read_csv(metadata_file, parse_dates=['recording_start_time','recording_stop_time','enroll_time'])
patients = 'ieeg'
patient_ids = ieeg_list

agg_baseline = []
agg_finetune = []
agg_prev = []
mean_diff = []
mean_perc_diff = []
full_baseline = []
full_finetune = []
for rid in patient_ids:
    print(rid)
    # load INFO
    feat_setting = {'win':1,'stride':1}
    base_model_name = 'oncet_base_model'
    meta_info = metadata[metadata['RID']==rid].iloc[0]

    # load push events
    push_events = pd.read_csv(f'annots/{rid}/patient_events.csv', parse_dates=['Start','Stop']).iloc[1:]
    if not push_events.empty:
        push_times = (push_events['Start']-meta_info['recording_start_time']+pd.Timedelta(seconds=60)).dt.total_seconds()/3600/24
    else:
        push_times = np.array([])
        
    # load base_model prediction
    base_model_pred = pd.read_csv(f'results/{rid}/{base_model_name}/prob/{base_model_name}.csv', parse_dates=['timestamp'])
    sz_events = extract_seiz_ranges(base_model_pred['label'].values)
    sz_events = [{'Start':base_model_pred['timestamp'].iloc[sz[0]]-pd.Timedelta(seconds=feat_setting['win']),
                  'Stop':base_model_pred['timestamp'].iloc[sz[1]]}
                  for sz in sz_events]
    sz_events = pd.DataFrame(sz_events)
    if not sz_events.empty:
        sz_times = (sz_events['Start']-meta_info['recording_start_time']).dt.total_seconds()/3600/24
    else:
        sz_times = np.array([])
    
    # version 1, plot predictions
    model = f'{rid}_{base_model_name}_lr5e-06_wd1e-03_{label_type}_{model_code}'
    pred_files = glob.glob(f'results/{rid}/{model}/prob/*.csv')
    if not pred_files:
        continue
    pred_files_df = pd.DataFrame({'filename':pred_files})
    pred_files_df['batch_ind'] = pred_files_df['filename'].apply(lambda x:int(x.split('/')[-1].split('_')[1][:-4]))
    pred_files_df = pred_files_df.sort_values(['batch_ind'])
    pred_files = pred_files_df['filename'].to_list()
    finetune_inds = pred_files_df['batch_ind'].to_list()#[int(f.split('/')[-1].split('_')[1]) for f in metric_files]
    finetune_inds.append(base_model_pred['batch_id'].max())
    
    remain_df = base_model_pred[['timestamp','batch_id','label']]
    prob_df = base_model_pred.drop(columns=['timestamp','batch_id','label'])
    n_ch = int(base_model_pred.shape[1]//2)
    prob_df = base_model_pred.iloc[:,:n_ch]
    prob = prob_df.quantile(0.8,axis=1).values#.mean(axis=1).values
    pred, perc_chan = ieeg_get_prediction(prob_df.values)
    base_model_pred = remain_df
    base_model_pred[prob_col] = prob
    base_model_pred[pred_col] = pred
    pred_sz_events = extract_seiz_ranges(base_model_pred[pred_col].values)
    pred_sz_events = [{'Start':base_model_pred['timestamp'].iloc[sz[0]]-pd.Timedelta(seconds=feat_setting['stride']),
                      'Stop':base_model_pred['timestamp'].iloc[sz[1]],
                      'is_sz':True if base_model_pred['label'].iloc[sz[0]-60:sz[1]+30].any() else False,
                      'avg_prob':base_model_pred[prob_col].iloc[sz[0]:sz[1]].mean()}
                      for sz in pred_sz_events]
    pred_sz_events = pd.DataFrame(pred_sz_events)
    pred_sz_times = (pred_sz_events['Start']-meta_info['recording_start_time']).dt.total_seconds()/3600/24
    sz_ranges = extract_seiz_ranges(base_model_pred['label'].values)
    sz_events_detected = np.array([np.any(base_model_pred[pred_col].iloc[start-30:end+60]==1) for start, end in sz_ranges])

    # build full pred
    full_pred = []
    comp_pred = []
    comp_base = []
    comp_prev = []
    full_pred.append(base_model_pred[base_model_pred['batch_id'].isin(list(range(finetune_inds[0]+1)))][['timestamp','batch_id','label',prob_col,pred_col]])
    for i, f in enumerate(pred_files):
        pred_df = pd.read_csv(f, parse_dates=['timestamp']).sort_values('timestamp')
        remain_df = pred_df[['timestamp','batch_id','label']]
        prob_df = pred_df.drop(columns=['timestamp','batch_id','label'])
        n_ch = int(pred_df.shape[1]//2)
        prob_df = pred_df.iloc[:,:n_ch]
        prob = prob_df.quantile(0.8,axis=1).values#.mean(axis=1).values
        pred, perc_chan = ieeg_get_prediction(prob_df.values)
        pred_df[prob_col] = prob
        pred_df[pred_col] = pred
        comp_pred.append(pred_df[pred_df['batch_id'] > finetune_inds[i]])
        comp_base.append(base_model_pred[base_model_pred['batch_id'] > finetune_inds[i]][['timestamp','batch_id','label',prob_col,pred_col]])
        if i == 0:
            comp_prev.append(base_model_pred[base_model_pred['batch_id'] > finetune_inds[i]])
        else:
            comp_prev.append(comp_pred[-1][comp_pred[-1]['batch_id'] > finetune_inds[i]])
        full_pred.append(pred_df[pred_df['batch_id'].isin(list(range(finetune_inds[i]+1,finetune_inds[i+1]+1)))])
    full_pred = pd.concat(full_pred, axis=0).sort_values('timestamp')

    pred_sz_events_ft = extract_seiz_ranges(full_pred[pred_col].values)
    pred_sz_events_ft = [{'Start':full_pred['timestamp'].iloc[sz[0]]-pd.Timedelta(seconds=feat_setting['stride']),
                      'Stop':full_pred['timestamp'].iloc[sz[1]],
                      'is_sz':True if full_pred['label'].iloc[sz[0]-60:sz[1]+30].any() else False,
                      'avg_prob':full_pred[prob_col].iloc[sz[0]:sz[1]].mean()}
                      for sz in pred_sz_events_ft]
    pred_sz_events_ft = pd.DataFrame(pred_sz_events_ft)
    pred_sz_times_ft = (pred_sz_events_ft['Start']-meta_info['recording_start_time']).dt.total_seconds()/3600/24
    sz_ranges_ft = extract_seiz_ranges(full_pred['label'].values)
    sz_events_detected_ft = np.array([np.any(full_pred[pred_col].iloc[start-30:end+60]==1) for start, end in sz_ranges_ft])

    fig, axs = plt.subplots(6,1,figsize=(15,10),sharex=True)
    axs[0].vlines(sz_times,ymin=0,ymax=1,color='k')
    axs[0].vlines((meta_info['enroll_time']-meta_info['recording_start_time']).total_seconds()/3600/24,ymin=0,ymax=1,color='r')
    axs[0].set_title('Seizure Times and Enroll Time')
    axs[1].vlines(push_times,ymin=0,ymax=1,color='blue')
    axs[1].set_title('Push Times')
    axs[2].vlines((np.array(finetune_inds[:-1])+1)*3/24,ymin=0,ymax=1,color='k',linestyle='--',alpha=0.5)
    axs[2].set_title('Finetune Times')
    
    axs[4].vlines(pred_sz_times[pred_sz_events['is_sz']==False],ymin=0,ymax=1,color='orange')
    axs[4].vlines(pred_sz_times[pred_sz_events['is_sz']],ymin=0,ymax=1,color='green')
    axs[4].set_title('Baseline Alarms')
    axs[5].vlines(pred_sz_times_ft[pred_sz_events_ft['is_sz']==False],ymin=0,ymax=1,color='orange')
    axs[5].vlines(pred_sz_times_ft[pred_sz_events_ft['is_sz']],ymin=0,ymax=1,color='green')
    axs[5].set_title('Finetuned Alarms')
    
    # build full metric trajectory 
    full_metric = []
    baseline_metrics = []
    for batch in full_pred['batch_id'].unique():
        batch_pred = full_pred[full_pred['batch_id']==batch]
        batch_label = batch_pred['label'].values
        batch_prob = batch_pred[prob_col].values
        batch_pred_vals = batch_pred[pred_col].values
        full_metric.append(compute_metrics(batch_label, batch_pred_vals, batch_prob, stride = feat_setting['stride']))
        # baseline
        batch_pred = base_model_pred[base_model_pred['batch_id']==batch]
        batch_label = batch_pred['label'].values
        batch_prob = batch_pred[prob_col].values
        batch_pred_vals = batch_pred[pred_col].values
        baseline_metrics.append(compute_metrics(batch_label, batch_pred_vals, batch_prob, stride = feat_setting['stride']))
    full_metric = pd.DataFrame(full_metric)
    baseline_metrics = pd.DataFrame(baseline_metrics) 
    baseline_metric_whole = compute_metrics(base_model_pred['label'].values, base_model_pred[pred_col].values, base_model_pred[prob_col].values, stride = feat_setting['stride'])
    finetune_metric_whole = compute_metrics(full_pred['label'].values, full_pred[pred_col].values, full_pred[prob_col].values, stride = feat_setting['stride'])
    
    comp_base_metrics = []
    comp_prev_metrics = []
    comp_finetune_metrics = []
    for base,prev,curr in zip(comp_base, comp_prev, comp_pred):
        comp_base_metrics.append(compute_metrics(base['label'].values, base[pred_col].values, base[prob_col].values, stride = feat_setting['stride']))
        comp_prev_metrics.append(compute_metrics(prev['label'].values, prev[pred_col].values, prev[prob_col].values, stride = feat_setting['stride']))
        comp_finetune_metrics.append(compute_metrics(curr['label'].values, curr[pred_col].values, curr[prob_col].values, stride = feat_setting['stride']))
    comp_base_metrics = pd.DataFrame(comp_base_metrics)
    comp_prev_metrics = pd.DataFrame(comp_prev_metrics)
    comp_finetune_metrics = pd.DataFrame(comp_finetune_metrics)
    tmp = comp_base_metrics[plot_vars].mean().to_frame().T
    tmp.index = [rid]
    agg_baseline.append(tmp)
    tmp = comp_prev_metrics[plot_vars].mean().to_frame().T
    tmp.index = [rid]
    agg_prev.append(tmp)
    tmp = comp_finetune_metrics[plot_vars].mean().to_frame().T
    tmp.index = [rid]
    agg_finetune.append(tmp)
    tmp = (comp_finetune_metrics[plot_vars]-comp_base_metrics[plot_vars]).median().to_frame().T
    tmp.index = [rid]
    mean_diff.append(tmp)
    tmp = ((comp_finetune_metrics[plot_vars]-comp_base_metrics[plot_vars])/comp_base_metrics[plot_vars]*100).median().to_frame().T
    tmp.index = [rid]
    mean_perc_diff.append(tmp)

    txt = '\n'.join([f'{metric_labels[var]}: {baseline_metric_whole[var]:.3f}' for var in ['auroc_sample','auprc_sample','recall_event','f1_event','fp']])
    axs[4].text(1.0, 0.1, txt, transform=axs[4].transAxes, fontsize=6,color="k")
    txt = '\n'.join([f'{metric_labels[var]}: {finetune_metric_whole[var]:.3f}' for var in ['auroc_sample','auprc_sample','recall_event','f1_event','fp']])
    axs[5].text(1.0, 0.1, txt, transform=axs[5].transAxes, fontsize=6,color="k")
    axs[3].plot(baseline_metrics.index.astype(int)*3/24, baseline_metrics['fp'].values,label='baseline',color=custom_palette[1], marker='o', markerfacecolor='white')
    axs[3].plot(full_metric.index.astype(int)*3/24, full_metric['fp'].values, label='finetuned', color=custom_palette[2],marker='o', markerfacecolor='white')
    axs[3].set_ylabel(metric_labels['fp'])
    axs[3].legend(loc='upper left')
    axs[5].set_xlabel("Length of Stay (days)")
    fig.savefig(f'{out_dir}/full_trajectory/{rid}_adjth_{adjth}_calibration_{calibrate}_{label_type}_{model_code}.svg',dpi=300)
    plt.close()
    
    baseline_tmp = pd.DataFrame([baseline_metric_whole])[plot_vars]#baseline_metrics[plot_vars].mean().to_frame().T
    baseline_tmp.index = [rid]
    full_baseline.append(baseline_tmp)
    finetune_tmp = pd.DataFrame([finetune_metric_whole])[plot_vars]#full_metric[plot_vars].mean().to_frame().T
    finetune_tmp.index = [rid]
    full_finetune.append(finetune_tmp)
    
    # the agg on test data only
    
full_baseline = pd.concat(full_baseline)
full_finetune = pd.concat(full_finetune)
agg_baseline = pd.concat(agg_baseline)
agg_finetune = pd.concat(agg_finetune)
agg_prev = pd.concat(agg_prev)
mean_diff = pd.concat(mean_diff)
mean_perc_diff = pd.concat(mean_perc_diff)
full_diff = full_finetune-full_baseline
full_perc_diff = ((full_finetune-full_baseline)/full_baseline*100)


In [ ]:
# save scalp results to concatenate with ieeg results
with open(f'{out_dir}/ieeg_results_{label_type}.pkl','wb') as f:
    pickle.dump((full_baseline, full_finetune, mean_diff, mean_perc_diff, full_diff, full_perc_diff, agg_baseline, agg_finetune, agg_prev), f)

### Combine scalp and ieeg results

In [ ]:
for label_type in ['patient', 'databricks']:
    # save scalp results to concatenate with ieeg results
    with open(f'{out_dir}/scalp_results_{label_type}.pkl','rb') as f:
        result = pickle.load(f)
    with open(f'{out_dir}/ieeg_results_{label_type}.pkl','rb') as f:
        ieeg_result = pickle.load(f)
    
    comp_result = []
    for r, ir in zip(result,ieeg_result):
        tmp = pd.concat([r,ir])
        tmp = tmp.sort_index(key=lambda x: x.map(patient_rank))
        comp_result.append(tmp)
    
    full_baseline, full_finetune, mean_diff, mean_perc_diff, full_diff, full_perc_diff, agg_baseline, agg_finetune, agg_prev = comp_result

    full_baseline.to_csv(f'{out_dir}/supp_baseline_performance_{label_type}.csv')
    full_finetune.to_csv(f'{out_dir}/supp_full_finetune_performance_{label_type}.csv')
    mean_diff.to_csv(f'{out_dir}/supp_mean_diff_{label_type}.csv')
    mean_perc_diff.to_csv(f'{out_dir}/supp_mean_perc_diff_{label_type}.csv')
    full_diff.to_csv(f'{out_dir}/supp_full_diff_{label_type}.csv')
    full_perc_diff.to_csv(f'{out_dir}/supp_full_perc_diff_{label_type}.csv')
    
    summary(full_baseline).to_csv(f'{out_dir}/supp_summary_baseline_{label_type}.csv')
    summary(full_finetune).to_csv(f'{out_dir}/supp_summary_full_finetune_{label_type}.csv')
    summary(mean_diff).to_csv(f'{out_dir}/supp_summary_mean_diff_{label_type}.csv')
    mean_perc_diff = mean_perc_diff.replace(np.inf,0)
    mean_perc_diff = mean_perc_diff.replace(np.nan,0)
    summary(mean_perc_diff).to_csv(f'{out_dir}/supp_summary_mean_perc_diff_{label_type}.csv')
    summary(full_diff).to_csv(f'{out_dir}/supp_summary_full_diff_{label_type}.csv')
    full_perc_diff = full_perc_diff.replace(np.inf,0)
    full_perc_diff = full_perc_diff.replace(np.nan,0)
    summary(full_perc_diff).to_csv(f'{out_dir}/supp_summary_full_perc_diff_{label_type}.csv')
    

In [ ]:
for label_type in ['patient', 'databricks']:
    # save scalp results to concatenate with ieeg results
    with open(f'{out_dir}/scalp_results_{label_type}.pkl','rb') as f:
        result = pickle.load(f)
    with open(f'{out_dir}/ieeg_results_{label_type}.pkl','rb') as f:
        ieeg_result = pickle.load(f)
    
    comp_result = []
    for r, ir in zip(result,ieeg_result):
        tmp = pd.concat([r,ir])
        tmp = tmp.sort_index(key=lambda x: x.map(patient_rank))
        comp_result.append(tmp)
    
    full_baseline, full_finetune, mean_diff, mean_perc_diff, full_diff, full_perc_diff, agg_baseline, agg_finetune, agg_prev = comp_result

    full_baseline['patient']=full_baseline.index
    plot_paired_line(full_baseline, full_finetune, 
                     variables=plot_vars, figsize = (10,4), save_path=f'{out_dir}/full_finetune_vs_baseline_{label_type}')
    #violin plot + scatter dots
    mean_diff = mean_diff.reset_index().rename(columns={'index':'patient'})
    results_long = pd.melt(
        mean_diff.loc[~mean_diff['patient'].isin(rm_patients)],
        id_vars='patient',
        var_name="metric",        # new column with former column names
        value_name="value"           # new column with values
    )
    fig,ax=plot_mean_diff_overlay(results_long, sep_axis_fp=True, variables = plot_vars, ieeg=True, ieeg_col='patient')
    fig.savefig(f'{out_dir}/mean_diff_{label_type}.svg')

### Statistical Analysis

In [ ]:
all_mean_diff = []
all_full = []
all_full_diff = []
for label_type in ['patient', 'databricks']:
    # save scalp results to concatenate with ieeg results
    with open(f'{out_dir}/scalp_results_{label_type}.pkl','rb') as f:
        result = pickle.load(f)
    with open(f'{out_dir}/ieeg_results_{label_type}.pkl','rb') as f:
        ieeg_result = pickle.load(f)
    
    comp_result = []
    for r, ir in zip(result,ieeg_result):
        tmp = pd.concat([r,ir])
        tmp = tmp.sort_index(key=lambda x: x.map(patient_rank))
        comp_result.append(tmp)
    
    tmp_full_baseline, tmp_full_finetune, tmp_mean_diff, tmp_mean_perc_diff, tmp_full_diff, tmp_full_perc_diff, _, _, _ = comp_result
    tmp_mean_diff['annot'] = label_type
    all_mean_diff.append(tmp_mean_diff)
    tmp_full_diff['annot'] = label_type
    all_full_diff.append(tmp_full_diff)
    tmp_full_baseline['model']='baseline'
    tmp_full_baseline['annot']=label_type
    all_full.append(tmp_full_baseline)
    tmp_full_finetune['model']='finetune'
    tmp_full_finetune['annot']=label_type
    all_full.append(tmp_full_finetune)
    
all_full = pd.concat(all_full)   
all_full_diff = pd.concat(all_full_diff)
all_mean_diff = pd.concat(all_mean_diff)
all_full_pivoted = all_full.pivot()

In [ ]:
from scipy import stats
def wilcoxon_one_sample_effects_and_ci(
    x,
    y=None,
    n_boot=1000,
    seed=42,
    ci=0.95,
    direction='greater'
):
    """
    diff: 1D array of paired differences (e.g. finetuned - baseline), one value per patient.
    Wilcoxon signed-rank tests H0: differences are symmetric about 0 (often used as a robust
    check on "typical" shift; interpret with median + bootstrap CI).
    """
    if y is None:
        diff = x
    else:
        diff = x-y
    diff = np.asarray(diff, dtype=float)
    diff = diff[~np.isnan(diff)]
    n = diff.size
    if n < 3:
        raise ValueError("Need at least 3 non-NaN differences for a stable Wilcoxon.")
    # zero_method='pratt' ignores pairs with diff==0 in ranking; 'wilcox' drops zeros entirely.
    wres = stats.wilcoxon(diff, alternative=direction, zero_method="wilcox",  method="auto")
    # Rank-biserial correlation (matched-pairs), common effect size for Wilcoxon in psychology/ML:
    # r_rb = sum(positive ranks - sum(negative ranks)) / sum(all ranks)
    # Implemented via signed ranks (exclude zeros for 'wilcox'-style; use pratt consistently).
    signs = np.sign(diff)
    absd = np.abs(diff)
    mask = absd > 0  # ranks only for non-zero |diff|
    ranks = stats.rankdata(absd[mask])
    sgn = signs[mask]
    sum_r_plus = ranks[sgn > 0].sum()
    sum_r_minus = ranks[sgn < 0].sum()
    denom = ranks.sum()
    r_rank_biserial = (sum_r_plus - sum_r_minus) / denom if denom > 0 else np.nan
    # Normal-approximation Z-based r (legacy "common language" effect size): r = Z / sqrt(N)
    # N here = number of signed ranks used (non-zero diffs) when using pratt/wilcox variants.
    # scipy doesn't expose Z directly; approximate from two-sided p (not ideal but sometimes reported).
    # Prefer r_rank_biserial above; we skip Z-based r to avoid misleading Z from discrete p.
    # Bootstrap CIs for mean(diff) and median(diff) — resample PATIENTS (rows).
    rng = np.random.default_rng(seed)
    alpha = 1 - ci
    boot_mean = np.empty(n_boot)
    boot_med = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        d = diff[idx]
        boot_mean[b] = d.mean()
        boot_med[b] = np.median(d)
    ci_mean = np.quantile(boot_mean, [alpha / 2, 1 - alpha / 2])
    ci_med = np.quantile(boot_med, [alpha / 2, 1 - alpha / 2])
    out = {
        "n_patients": int(n),
        "median_diff": float(np.median(diff)),
        "mean_diff": float(np.mean(diff)),
        "iqr_diff": tuple(np.percentile(diff, [25, 75]).astype(float)),
        "wilcoxon_statistic": float(wres.statistic),
        "wilcoxon_pvalue": float(wres.pvalue),
        "rank_biserial_r": float(r_rank_biserial),
        f"bootstrap_mean_{int(ci*100)}ci": tuple(ci_mean.astype(float)),
        f"bootstrap_median_{int(ci*100)}ci": tuple(ci_med.astype(float)),
    }
    return out


In [ ]:
mean_diff_out = []
full_diff_out = []
for label_type in ['patient','databricks']:
    for var in ['fp','f1_event']:
        if var == 'fp':
            direction = 'less'
        else:
            direction = 'greater'
        x = all_mean_diff[all_mean_diff['annot']==label_type][var].values
        mean_diff_out.append(wilcoxon_one_sample_effects_and_ci(x,direction=direction))
        x = all_full_diff[all_full_diff['annot']==label_type][var].values
        full_diff_out.append(wilcoxon_one_sample_effects_and_ci(x,direction=direction))
mean_diff_out = pd.DataFrame(mean_diff_out)
full_diff_out = pd.DataFrame(full_diff_out)
mean_diff_out[['label_type','var']] = [(label_type, var) for label_type in ['patient','databricks'] for var in ['fp','f1_event']]
full_diff_out[['label_type','var']] = [(label_type, var) for label_type in ['patient','databricks'] for var in ['fp','f1_event']]
mean_diff_out['setting'] = 'mean'
full_diff_out['setting'] = 'full'

In [ ]:
x = all_full[(all_full['annot']=='patient')&(all_full['model']=='finetune')]
y = all_full[(all_full['annot']=='databricks')&(all_full['model']=='finetune')]
x = x[x.index.isin(y.index)]
label_comp_out = []
for var in ['fp','f1_event']:
    if var == 'fp':
        direction = 'less'
    else:
        direction = 'greater'
    label_comp_out.append(wilcoxon_one_sample_effects_and_ci(x[var].values,y[var].values))
label_comp_out = pd.DataFrame(label_comp_out)
label_comp_out['var'] = ['fp','f1_event']
label_comp_out['setting'] = 'comp'
all_out = pd.concat([mean_diff_out,full_diff_out,label_comp_out])
from statsmodels.stats.multitest import multipletests
reject, pvals_bh, _, _ = multipletests(all_out['wilcoxon_pvalue'].values, alpha=0.05, method="fdr_bh")
all_out['corrected_pvalue'] = pvals_bh
all_out.to_csv(f'{out_dir}/stats.csv')

In [ ]:
all_out_format = all_out.copy()
all_out_format[['median_diff','mean_diff','wilcoxon_pvalue','rank_biserial_r','corrected_pvalue']] = all_out_format[['median_diff','mean_diff','wilcoxon_pvalue','rank_biserial_r','corrected_pvalue']].round(3)
all_out_format['iqr_diff'] = all_out_format['iqr_diff'].apply(lambda x: f'[{x[0].round(3)},{x[1].round(3)}]')
all_out_format['bootstrap_mean_95ci'] = all_out_format['bootstrap_mean_95ci'].apply(lambda x: f'[{x[0].round(3)},{x[1].round(3)}]')
all_out_format['bootstrap_median_95ci'] = all_out_format['bootstrap_median_95ci'].apply(lambda x: f'[{x[0].round(3)},{x[1].round(3)}]')
all_out_format.to_csv(f'{out_dir}/stats_format.csv')

In [ ]:
all_out_format

### Accuracy of patient push button events

In [ ]:
def events_overlap(e1, e2, pre_gap=30, post_gap=60):
    """
    e1, e2: (start, end)
    Returns True if events overlap OR gap <= max_gap
    """
    s1, e1 = e1
    s1 = s1-np.timedelta64(pre_gap,'s')
    e1 = e1+np.timedelta64(post_gap, 's')
    s2, e2 = e2

    # Overlap
    if min(e1, e2) >= max(s1, s2):
        return True

    return False


def match_events(true_events, pred_events, max_gap=10):
    """
    Greedy 1-to-1 matching.
    Returns matched true indices and pred indices.
    """
    matched_true = set()
    matched_pred = set()

    for j, p in enumerate(pred_events):
        for i, t in enumerate(true_events):
            if events_overlap(t, p, pre_gap=30, post_gap=60):
                matched_true.add(i)
                matched_pred.add(j)

    return matched_true, matched_pred
    
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
def draw_heatmap(data, annot_labels,
                 cmap=None,
                 fmt='',
                 cbar=True,
                 vmin=0,
                 vmax=1,
                 xticklabels=None,
                 yticklabels=None,
                 title=None,
                ax=None):
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list(
            "bluewhite",
            ["#ffffff","#025086"]
        )
        cmap.set_bad(color="#E5E5E5")   # NaN color
        cmap.set_under("#E5E5E5")
    data = np.asarray(data)
    annot_labels = np.asarray(annot_labels)

    if data.shape != annot_labels.shape:
        raise ValueError("data and annot_labels must have same shape")
    
    mask = data < 0
    if ax is None:
        fig, ax = plt.subplots(figsize=(0.7+data.shape[1],8))
    
    ax = sns.heatmap(
        data,
        annot=annot_labels,
        fmt=fmt,
        cmap=cmap,
        cbar=False,
        vmin=vmin,
        vmax=vmax,
        mask=~mask, 
        xticklabels=xticklabels,
        yticklabels=yticklabels,
        linewidths=1.5,
        linecolor='white',
        annot_kws={"fontsize": 12, "fontweight":'medium'}
    )

    ax = sns.heatmap(
        data,
        annot=annot_labels,
        fmt=fmt,
        cmap=cmap,
        cbar=cbar,
        vmin=vmin,
        vmax=vmax,
        mask=mask,
        xticklabels=xticklabels,
        yticklabels=yticklabels,
        linewidths=1.5,
        linecolor="white",
        annot_kws={"fontsize": 16, "fontweight":'medium'},
        cbar_kws={"fraction": 0.1}
    )
    # Gray borders only for non-NaN cells
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if data[i, j] != -1:
                ax.add_patch(
                    Rectangle(
                        (j, i), 1, 1,
                        fill=False,
                        edgecolor="#E5E5E5",
                        linewidth=1.5
                    )
                )

    if cbar:
        cbar = ax.collections[-1].colorbar
        cbar.ax.tick_params(labelsize=16)

    ax.set_yticklabels(ax.get_yticklabels(), fontsize=16, fontweight='medium', rotation=0)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=16, fontweight='medium', rotation=0)


    if title is not None:
        ax.set_title(title, fontsize=16, fontweight='bold')
        
    # Make the outer border (spines) visible
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(1.5) # Set thickness
        spine.set_edgecolor('#E5E5E5') # Set color

    plt.tight_layout()
    return ax


In [ ]:
metadata = pd.read_csv(metadata_file, parse_dates=['recording_start_time','recording_stop_time','enroll_time'])
patients = 'all'
if patients == 'all':
    patient_ids = sorted(os.listdir('../rawdata'))
elif patients == 'scalp':
    patient_ids = sorted(os.listdir('../rawdata'))
    patient_ids = [p for p in patient_ids if p not in ieeg_list]
elif patients == 'ieeg':
    patient_ids = ieeg_list
else:
    patient_ids = patients.split(',')

patient_ids = [p for p in patient_ids if p not in rm_patients]
patient_ids = sorted(patient_ids, key=lambda x: patient_rank[x])
all_results = []
all_push_true = []
all_push_pred = []
for rid in patient_ids:
    print(rid)
    
    # load INFO
    with open(f"INFO/{rid}.pkl", "rb") as f:
        info = pickle.load(f)
    if info['studyType'] == 'eeg':
        feat_setting = {'win':10,'stride':2}
        base_model_name = 'base_model'
    elif info['studyType'] == 'ieeg':
        feat_setting = {'win':1,'stride':1}
        base_model_name = 'oncet_base_model'
    meta_info = metadata[metadata['RID']==rid].iloc[0]
    
    # load events
    sz_events = pd.read_csv(f'annots/{rid}/seizure_events.csv', parse_dates=['Start','Stop'])
    sz_events['Start'] = sz_events['Start']
    sz_events['Stop'] = sz_events['Stop']
    sz_events = sz_events[['Start','Stop']].to_numpy()
    push_events = pd.read_csv(f'annots/{rid}/patient_events.csv', parse_dates=['Start','Stop']).iloc[1:][['Start','Stop']].to_numpy()
    matched_true, matched_pred = match_events(sz_events, push_events)
    all_push_true.append([len(matched_true),len(sz_events)-len(matched_true)])
    all_push_pred.append([len(matched_pred),len(push_events)-len(matched_pred)])
    sens = len(matched_true)/len(sz_events) if len(sz_events) > 0 else np.nan
    precision = len(matched_pred)/len(push_events) if len(push_events) > 0  else np.nan
    f1 = 2 * precision * sens / (precision + sens) if len(sz_events)+len(push_events) > 0 else np.nan
    if (len(sz_events)+len(push_events) > 0) and np.isnan(f1):
        f1 = 0.0
    all_results.append({'patient':rid,'recall_event':sens,'precision_event':precision,'f1_event':f1})
results = pd.DataFrame(all_results)
all_push_true = np.array(all_push_true)
all_push_pred = np.array(all_push_pred)
results.to_csv(f'{out_dir}/supp_push_button_accuracy.csv',index=False)

In [ ]:
results_long = pd.melt(
    results,
    id_vars=['patient'],
    var_name="metric",        # new column with former column names
    value_name="value"           # new column with values
)
fig,ax=plot_mean_diff_overlay(results_long, variables = ['recall_event','precision_event','f1_event'],ieeg=True, ieeg_col='patient')
fig.savefig(f'{out_dir}/supp_fig_push_button_accuracy.svg',dpi=300)

In [ ]:
summary_tbl = summary(results[['recall_event','precision_event','f1_event']])
summary_tbl.to_csv(f'{out_dir}/supp_summary_push_button_accuracy.csv')

In [ ]:
summary_tbl

### Databricks events

In [ ]:
# system alert's sensitivity and precision
alerts = pd.read_csv('databricks/pioneer_seizure_events_and_responses.csv', 
                     dtype={'databricks_seizure_event_id':str,'annotation_id':str},
                    parse_dates = ['notification_sent_at_et','patient_response_at_et'])

alerts = alerts[['seizure_event_id','databricks_event_id','notification_sent_at_et','ai_message_content','annotation_id','patient_response','response_classification','patient_response_at_et']]
alerts['patientGuid'] = alerts['databricks_event_id'].apply(lambda x: x.split('_')[0])
alerts = alerts[alerts['patientGuid'].isin(metadata['patientGuid'])]
alerts = alerts.drop_duplicates(subset=['seizure_event_id'])
notifications = alerts[~alerts['notification_sent_at_et'].isna()]
notifications.to_csv('databricks/notifications.csv',index=False)

In [ ]:
notifications = pd.read_csv('databricks/notifications.csv',
                           dtype={'databricks_seizure_event_id':str,'annotation_id':str},
                            parse_dates = ['notification_sent_at_et','patient_response_at_et'])
notifications['start_stamp'] = notifications['databricks_event_id'].apply(lambda x:x.split('_')[1]).astype(int)

In [ ]:
databricks_detections = pd.read_csv('databricks/pioneer_sz_events_annots_new.csv')
databricks_detections['start_time'] = databricks_detections['start_time'].apply(lambda x: pd.Timestamp(x).tz_localize(None))
databricks_detections['stop_time'] = databricks_detections['stop_time'].apply(lambda x: pd.Timestamp(x).tz_localize(None))

In [ ]:
# databricks V2
metadata = pd.read_csv(metadata_file, parse_dates=['recording_start_time','recording_stop_time','enroll_time'])
patients = 'all'
if patients == 'all':
    patient_ids = sorted(os.listdir('../rawdata'))
elif patients == 'scalp':
    patient_ids = sorted(os.listdir('../rawdata'))
    patient_ids = [p for p in patient_ids if p not in ieeg_list]
elif patients == 'ieeg':
    patient_ids = ieeg_list
else:
    patient_ids = patients.split(',')
patient_ids = [p for p in patient_ids if p not in rm_patients]
patient_ids = sorted(patient_ids, key=lambda x: patient_rank[x])

all_results = []
all_alert_true = []
all_alert_pred = []

for rid in patient_ids:
    print(rid)
    if rid in ['P001','P002']:
        all_results.append({'patient':rid,'alert_recall':np.nan,'alert_precision':np.nan})
        all_alert_true.append([np.nan, np.nan])
        all_alert_pred.append([np.nan, np.nan])
        continue
    
    # load INFO
    with open(f"INFO/{rid}.pkl", "rb") as f:
        info = pickle.load(f)
    if info['studyType'] == 'eeg':
        feat_setting = {'win':10,'stride':2}
        base_model_name = 'base_model'
    elif info['studyType'] == 'ieeg':
        feat_setting = {'win':1,'stride':1}
        base_model_name = 'oncet_base_model'
    meta_info = metadata[metadata['RID']==rid].iloc[0]
    guid = meta_info['patientGuid']

    # load events
    sz_events = pd.read_csv(f'annots/{rid}/seizure_events.csv', parse_dates=['Start','Stop'])
    sz_events['Start'] = sz_events['Start']#-pd.Timedelta(seconds=30)
    sz_events['Stop'] = sz_events['Stop']#+pd.Timedelta(seconds=60)
    sz_events = sz_events[['Start','Stop']].to_numpy()
    try:
        alert_pat = notifications[notifications['patientGuid']==guid]
    except Exception as e:
        print(e)
        continue
    databricks_events = databricks_detections[(databricks_detections['patient_id']==guid)&(databricks_detections['start_stamp'].isin(alert_pat['start_stamp']))]
    match_true, match_pred = match_events(sz_events, databricks_events[['start_time','stop_time']].to_numpy())
    all_alert_true.append([len(match_true), len(sz_events)-len(match_true)])
    all_alert_pred.append([len(match_pred), len(databricks_events)-len(match_pred)])
    databricks_sens = len(match_true)/len(sz_events) if len(sz_events)>0 else np.nan
    databricks_precision = len(match_pred)/len(databricks_events) if len(databricks_events)>0 else np.nan
    
    all_results.append({'patient':rid,'alert_recall':databricks_sens,'alert_precision':databricks_precision})
    
all_alert_true = np.array(all_alert_true)
all_alert_pred = np.array(all_alert_pred)
results = pd.DataFrame(all_results)
results.to_csv(f'{out_dir}/supp_databricks_alert_accuracy.csv',index=False)

In [ ]:
# databricks V2
metadata = pd.read_csv(metadata_file, parse_dates=['recording_start_time','recording_stop_time','enroll_time'])
patients = 'all'
if patients == 'all':
    patient_ids = sorted(os.listdir('../rawdata'))
elif patients == 'scalp':
    patient_ids = sorted(os.listdir('../rawdata'))
    patient_ids = [p for p in patient_ids if p not in ieeg_list]
elif patients == 'ieeg':
    patient_ids = ieeg_list
else:
    patient_ids = patients.split(',')
patient_ids = [p for p in patient_ids if p not in rm_patients]
patient_ids = sorted(patient_ids, key=lambda x: patient_rank[x])

all_results = []
all_databricks_true = []
all_databricks_pred = []
all_patient_cm = []
all_cm_true = []#np.zeros((3,2))
all_cm_pred = []
all_label = []

for rid in patient_ids:
    print(rid)
    
    # load INFO
    with open(f"INFO/{rid}.pkl", "rb") as f:
        info = pickle.load(f)
    if info['studyType'] == 'eeg':
        feat_setting = {'win':10,'stride':2}
        base_model_name = 'base_model'
    elif info['studyType'] == 'ieeg':
        feat_setting = {'win':1,'stride':1}
        base_model_name = 'oncet_base_model'
    meta_info = metadata[metadata['RID']==rid].iloc[0]

    # load events
    sz_events = pd.read_csv(f'annots/{rid}/seizure_events.csv', parse_dates=['Start','Stop'])
    sz_events['Start'] = sz_events['Start']#-pd.Timedelta(seconds=30)
    sz_events['Stop'] = sz_events['Stop']#+pd.Timedelta(seconds=60)
    sz_events = sz_events[['Start','Stop']].to_numpy()
    try:
        databricks_events = pd.read_csv(f'annots/{rid}/databricks_events.csv', parse_dates=['start_time','stop_time'])
    except:
        all_cm_true.append([np.nan, np.nan])
        all_cm_pred.append([np.nan, np.nan])
        all_patient_cm.append(np.nan*np.zeros([2,3]))
        all_databricks_true.append([np.nan, np.nan])
        all_databricks_pred.append([np.nan, np.nan])
        continue
    databricks_events['start_time'] = databricks_events['start_time'].apply(lambda x: pd.Timestamp(x).tz_localize(None))
    databricks_events['stop_time'] = databricks_events['stop_time'].apply(lambda x: pd.Timestamp(x).tz_localize(None))
    databricks_events = databricks_events[~databricks_events['seizure_event_id'].isna()]
    databricks_events = databricks_events[['start_time','stop_time','classified_response']]#.to_numpy()
    match_true, match_pred = match_events(sz_events, databricks_events[['start_time','stop_time']].to_numpy())
    all_databricks_true.append([len(match_true), len(sz_events)-len(match_true)])
    all_databricks_pred.append([len(match_pred), len(databricks_events)-len(match_pred)])
    databricks_label = np.array([1 if i in match_pred else 0 for i in range(len(databricks_events))])
    databricks_sens = len(match_true)/len(sz_events) if len(sz_events)>0 else np.nan
    databricks_precision = len(match_pred)/len(databricks_events) if len(databricks_events)>0 else np.nan
    cm = []
    for i in [1,0]:
        pred_events = databricks_events[databricks_label==i]
        response_count = []
        for response in ['yes', 'no','uncertain']:
            response_count.append((pred_events['classified_response']==response).sum())
        cm.append(response_count)
    cm = np.array(cm)
    num_event = cm.sum(axis=0)
    sens = cm[0,0]/np.sum(cm[0,:]) if np.sum(cm[0,:]) > 0 else np.nan
    precision = cm[0,0]/np.sum(cm[:,0]) if np.sum(cm[:,0]) > 0 else np.nan
    f1 = 2 * precision * sens / (precision + sens) if len(sz_events) + len(databricks_events) > 0 else np.nan
    if len(sz_events) + len(databricks_events) > 0 and f1 == np.nan:
        f1 = 0.0
    npv = cm[1,1]/np.sum(cm[:,1]) if np.sum(cm[:,1]) > 0 else np.nan
    yes_events = databricks_events[databricks_events['classified_response']=='yes']
    match_true, match_pred = match_events(sz_events, yes_events[['start_time','stop_time']].to_numpy())
    overall_sens = len(match_true)/len(sz_events) if len(sz_events)>0 else np.nan
    overall_precision = len(match_pred)/len(yes_events) if len(yes_events)>0 else np.nan

    cm = np.array(cm)
    print(cm)
    all_patient_cm.append(cm)
    all_cm_true.append([cm[0,0],len(sz_events)-cm[0,0]])
    all_cm_pred.append([cm[0,0],len(yes_events)-cm[0,0]])
    
    
    all_results.append({'patient':rid,'n_yes':num_event[0],'n_no':num_event[1],'n_unclear':num_event[2],
                        'databricks_recall':databricks_sens,'databricks_precision':databricks_precision,
                        'patient_recall':sens,'patient_precision':precision,'patient_npv':npv,'patient_f1_event':f1,
                       'overall_recall':overall_sens, 'overall_precision':overall_precision})
    

all_patient_cm = np.array(all_patient_cm)
all_databricks_true = np.array(all_databricks_true)
all_databricks_pred = np.array(all_databricks_pred)
all_cm_true = np.array(all_cm_true)
all_cm_pred = np.array(all_cm_pred)
results = pd.DataFrame(all_results)
results.to_csv(f'{out_dir}/supp_patient_annotation_accuracy.csv',index=False)

In [ ]:
titles = ["Push button","Automatic alerts","Patient annotation","Platform overall"]
fig = plt.figure(figsize=(16, 10))

gs = fig.add_gridspec(1, 4, width_ratios=[2, 2, 3, 2.3],wspace=0.3)

for i in range(4):
    title = titles[i]
    ax = fig.add_subplot(gs[i])
    if i == 0:
        all_cm_true_plot = np.vstack([all_push_true,all_push_true.sum(axis=0)])
        all_cm_pred_plot = np.vstack([all_push_pred,all_push_pred.sum(axis=0)])
        plot_data = np.vstack([all_cm_true_plot[:,0]/all_cm_true_plot.sum(axis=1), all_cm_pred_plot[:,0]/all_cm_pred_plot.sum(axis=1)]).T
        plot_data = np.nan_to_num(plot_data, nan=0)
        plot_label = np.vstack([[f"{row[0]}/{row.sum()}" if row.sum() > 0 else "" for row in all_cm_true_plot],[f"{row[0]}/{row.sum()}"  if row.sum() > 0 else "" for row in all_cm_pred_plot]]).T
        ax = draw_heatmap(plot_data, plot_label, xticklabels=['Sens.','Prec.'], yticklabels=[f'P{i:03d}'for i in range(1,14)]+['Sum'], title=title, cbar=False,ax=ax)
    elif i == 1:
        all_cm_true_plot = np.vstack([all_alert_true,np.nansum(all_alert_true,axis=0)])
        all_cm_pred_plot = np.vstack([all_alert_pred,np.nansum(all_alert_pred,axis=0)])
        plot_data = np.vstack([all_cm_true_plot[:,0]/all_cm_true_plot.sum(axis=1), all_cm_pred_plot[:,0]/all_cm_pred_plot.sum(axis=1)]).T
        plot_data[all_cm_true_plot.sum(axis=1)==0,0]=0
        plot_data[all_cm_pred_plot.sum(axis=1)==0,1]=0
        plot_data = np.nan_to_num(plot_data, nan=-1)
        plot_label = np.vstack([[f"{int(row[0])}/{int(row.sum())}" if row.sum()>0 else "" for row in all_cm_true_plot],[f"{int(row[0])}/{int(row.sum())}" if row.sum()>0 else "" for row in all_cm_pred_plot]]).T
        ax = draw_heatmap(plot_data, plot_label, xticklabels=['Sens.','Prec.'], yticklabels=[],title=title, cbar=False,ax=ax)
        ax.set_yticklabels([])
    elif i == 2:
        all_patient_cm_plot = np.concatenate([all_patient_cm,np.nansum(all_patient_cm,axis=0).reshape(1,2,3)],axis=0)
        ratio_data = np.array([[all_patient_cm_plot[:,0,0],all_patient_cm_plot[:,0,:].sum(axis=-1)], 
                       [all_patient_cm_plot[:,0,0],all_patient_cm_plot[:,:,0].sum(axis=-1)],
                       [all_patient_cm_plot[:,1,1],all_patient_cm_plot[:,:,1].sum(axis=-1)]]).transpose([0,2,1])
        plot_data = (ratio_data[:,:,0]/ratio_data[:,:,1]).T
        plot_data[ratio_data[:,:,1].T==0]=0
        plot_data = np.nan_to_num(plot_data, nan=-1)
        def format_annot(data):
            return [f"{int(row[0])}/{int(row[1])}" if row[1]>0 else "" for row in data]
        plot_label = np.array([format_annot(x) for x in ratio_data]).T
        ax = draw_heatmap(plot_data, plot_label, xticklabels=['Sens.','Prec.','NPV'], yticklabels=[],title=title, cbar=False,ax=ax)
        ax.set_yticklabels([])
    elif i == 3:
        all_cm_true_plot = np.vstack([all_cm_true,np.nansum(all_cm_true,axis=0)])
        all_cm_pred_plot = np.vstack([all_cm_pred,np.nansum(all_cm_pred,axis=0)])
        plot_data = np.vstack([all_cm_true_plot[:,0]/all_cm_true_plot.sum(axis=1), all_cm_pred_plot[:,0]/all_cm_pred_plot.sum(axis=1)]).T
        plot_data[all_cm_true_plot.sum(axis=1)==0,0]=0
        plot_data[all_cm_pred_plot.sum(axis=1)==0,1]=0
        plot_data = np.nan_to_num(plot_data, nan=-1)
        plot_label = np.vstack([[f"{int(row[0])}/{int(row.sum())}" if row.sum()>0 else "" for row in all_cm_true_plot],[f"{int(row[0])}/{int(row.sum())}" if row.sum()>0 else "" for row in all_cm_pred_plot]]).T
        ax = draw_heatmap(plot_data, plot_label, xticklabels=['Sens.','Prec.'], yticklabels=[],title=title, ax=ax)
fig.savefig(f'{out_dir}/supp_annotation_heatmap.svg')

In [ ]:
tmp = results[(results['n_yes']>0)|(results['n_no']>0)]
tmp = tmp.rename(columns={'overall_recall':'recall_event','overall_precision':'precision_event'})
results_long = pd.melt(
    tmp,
    id_vars=['patient'],
    var_name="metric",        # new column with former column names
    value_name="value"           # new column with values
)
fig,ax=plot_mean_diff_overlay(results_long, variables = ['recall_event','precision_event'], ieeg=True, ieeg_col='patient')
fig.savefig(f'{out_dir}/supp_fig_databricks_event_accuracy.svg',dpi=300)

In [ ]:
summary_tbl = summary(results[(results['n_yes']>0)|(results['n_no']>0)].drop(columns='patient'))
summary_tbl.to_csv(f'{out_dir}/supp_summary_databricks_event_accuracy.csv')